In [ ]:
!pip install -q peft transformers datasets evaluate bitsandbytes rouge_score huggingface_hub


import os                     
import numpy as np            
import logging                
from PIL import Image         
import torch                  
import random                 
import json                   

from datasets import load_dataset 
from peft import LoraConfig, get_peft_model 
from transformers import (
    PaliGemmaProcessor,             
    PaliGemmaForConditionalGeneration, 
    Trainer,                        
    TrainingArguments,              
    BitsAndBytesConfig,             
)
import evaluate               
from huggingface_hub import notebook_login 



logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)



if torch.cuda.is_available():
    device = torch.device("cuda")
    logger.info(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    logger.info("GPU not available, using CPU instead.")


logger.info("Loading the clevr_cogen_a_train dataset...")

full_subset = load_dataset("leonardPKU/clevr_cogen_a_train", split="train[:20%]")



split_datasets = full_subset.train_test_split(test_size=0.1, seed=42)


train_dataset = split_datasets["train"]
test_dataset = split_datasets["test"]

logger.info(f"Training dataset size: {len(train_dataset)}")
logger.info(f"Testing dataset size: {len(test_dataset)}")

/home/moein_salimi/users/babak/IdeaGeneration/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO:__main__:Using device: NVIDIA GeForce RTX 3090


INFO:__main__:Loading the clevr_cogen_a_train dataset...


INFO:__main__:Training dataset size: 12600


INFO:__main__:Testing dataset size: 1400


In [ ]:
model_id = "./paligemma-3b-mix-224"
logger.info(f"Loading processor from {model_id}...")


processor = PaliGemmaProcessor.from_pretrained(model_id)


bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
)


logger.info("Loading PaliGemma model in 8-bit precision...")


model = PaliGemmaForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=bnb_config,
)


logger.info("Configuring LoRA for efficient fine-tuning...")


lora_config = LoraConfig(
    r=64,
    lora_alpha=64,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "o_proj",
        "k_proj",
        "v_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)


logger.info("Trainable parameters after applying LoRA:")


model.print_trainable_parameters()

INFO:__main__:Loading processor from ./paligemma-3b-mix-224...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


INFO:__main__:Loading PaliGemma model in 8-bit precision...


INFO:accelerate.utils.modeling:We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).



Loading checkpoint shards:   0%|                                                                                                           | 0/3 [00:00<?, ?it/s]

...


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:21<00:00,  7.06s/it]


Loading checkpoint shards: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:21<00:00,  7.20s/it]


INFO:__main__:Configuring LoRA for efficient fine-tuning...


INFO:__main__:Trainable parameters after applying LoRA:


trainable params: 90,390,528 || all params: 3,013,857,008 || trainable%: 2.9992


In [ ]:
def preprocess_function(batch):
    questions = batch["problem"]
    images = batch["image"]
    answers = batch["solution"]

    processed_images = []
    texts_with_image = []

    for q, img in zip(questions, images):
        try:
            pil_img = img.convert("RGB").resize((224, 224))
            processed_images.append(pil_img)
            texts_with_image.append("<image> " + q)
        except Exception as e:
            logger.warning(f"Error processing an image, skipping it: {e}")
            processed_images.append(None)
            texts_with_image.append(None)

    valid_indices = [i for i, img in enumerate(processed_images) if img is not None]
    if not valid_indices:
        return {}

    processed_images = [processed_images[i] for i in valid_indices]
    texts_with_image = [texts_with_image[i] for i in valid_indices]
    valid_answers = [answers[i] for i in valid_indices]

    encoder_inputs = processor(
        images=processed_images,
        text=texts_with_image,
        padding="max_length",
        truncation=True,
        max_length=300,
        return_tensors="pt",
    )

    decoder_inputs = processor.tokenizer(
        text_target=valid_answers,
        padding="max_length",
        truncation=True,
        max_length=300,
        return_tensors="pt",
    )

    labels_ids = decoder_inputs["input_ids"].clone()
    padding_mask = decoder_inputs["attention_mask"] == 0
    labels_ids[padding_mask] = -100
    encoder_inputs["labels"] = labels_ids
    return encoder_inputs


logger.info("Applying preprocessing to the training dataset...")
processed_train_dataset = train_dataset.map(
    preprocess_function, batched=True, remove_columns=train_dataset.column_names
)

logger.info("Applying preprocessing to the testing dataset...")
processed_test_dataset = test_dataset.map(
    preprocess_function, batched=True, remove_columns=test_dataset.column_names
)

INFO:__main__:Applying preprocessing to the training dataset...



Map:   0%|                                                                                                                      | 0/12600 [00:00<?, ? examples/s]


Map:   8%|████████▍                                                                                                  | 1000/12600 [00:34<06:42, 28.83 examples/s]


Map:   8%|████████▍                                                                                                  | 1000/12600 [00:46<06:42, 28.83 examples/s]


Map:  16%|████████████████▉                                                                                          | 2000/12600 [01:07<05:56, 29.76 examples/s]

...


INFO:__main__:Applying preprocessing to the testing dataset...



Map:   0%|                                                                                                                       | 0/1400 [00:00<?, ? examples/s]


Map:  71%|█████████████████████████████████████████████████████████████████████████████▏                              | 1000/1400 [00:22<00:09, 44.23 examples/s]


Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1400/1400 [00:32<00:00, 42.46 examples/s]


Map: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1400/1400 [00:36<00:00, 38.51 examples/s]

In [ ]:
training_args = TrainingArguments(
    output_dir="./paligemma-clevr-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    logging_steps=10,
    learning_rate=1e-4,
    load_best_model_at_end=True,
    report_to="none",
    remove_unused_columns=False,
    fp16=True,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=processed_train_dataset,
    eval_dataset=processed_test_dataset,
    tokenizer=processor.tokenizer,
)

logger.info("Starting the fine-tuning process...")

trainer.train()
logger.info("Fine-tuning completed.")

/tmp/ipykernel_4082756/1548587602.py:27: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


INFO:__main__:Starting the fine-tuning process...


/home/moein_salimi/users/babak/IdeaGeneration/venv/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,0.088500,0.087241
200,0.114900,0.077644
300,0.071800,0.068535
400,0.108600,0.048810
500,0.026900,0.027775
600,0.035200,0.042240
700,0.093900,0.023377


...

/home/moein_salimi/users/babak/IdeaGeneration/venv/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


/home/moein_salimi/users/babak/IdeaGeneration/venv/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


/home/moein_salimi/users/babak/IdeaGeneration/venv/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


/home/moein_salimi/users/babak/IdeaGeneration/venv/lib/python3.10/site-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


INFO:__main__:Fine-tuning completed.
